### 0. Download AIRTLab Dataset

In [10]:
from pathlib import Path
BASE_DIR = Path.cwd().parent.parent
print(f"Base directory: {BASE_DIR}")

Base directory: d:\Myworkplace\Python\violence-movies


In [ ]:
# # downloads the AIRTLAB dataset for violence detection
# !mkdir -p ../../data/raw
# !git clone https://github.com/airtlab/A-Dataset-for-Automatic-Violence-Detection-in-Videos.git ../../data/raw

Cloning into './data/raw'...
remote: Enumerating objects: 376, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 376 (delta 3), reused 11 (delta 3), pack-reused 364 (from 1)
Receiving objects: 100% (376/376), 1.02 GiB | 35.26 MiB/s, done.
Resolving deltas: 100% (3/3), done.
Updating files: 100% (355/355), done.


### 1. Seperate AIRTLab into 3 folders ("high-level violence", "low-level violence", "non-violence")

In [ ]:
# Seperate AIRTLab into 3 folders ("high-level violence", "low-level violence", "non-violence")

from pathlib import Path
import shutil

BASE_DIR = Path.cwd().parent.parent

# ====== CONFIG ======
# Dataset root
ROOT = BASE_DIR / "data" / "raw" / "violence-detection-dataset"

# CSV files
CSV_NONVIOLENT = ROOT / "nonviolent-action-classes.csv"
CSV_VIOLENT    = ROOT / "violent-action-classes.csv"

# Output directory
OUT = BASE_DIR / "data" / "processed" / "violence-detection-dataset"

COPY_FILES = True   # True = copy, False = move
# ====================

# Labels from the paper table (make them lowercase)
NON_VIOLENCE_LABELS = {
    "handshake", "highfive", "hug", "jump", "walk", "greet",
    # dataset sometimes uses variants:
    "handgestures", "friendly punch"
}

LOW_LEVEL_LABELS = {
    "push", "slap", "stifle", "fight", "kick", "punch"
}

HIGH_LEVEL_LABELS = {
    "shoot", "stab", "club"
}

# We'll normalize by splitting on ",".
def normalize_actions(action_str: str) -> set[str]:
    return {a for a in action_str.split(",") if a}

def read_csv_map(csv_path: Path) -> dict[str, set[str]]:
    """
    Expected lines like:
    FILE; ACTION CLASSES
    1.mp4;hug,highfive
    2.mp4;highfive,greet
    """
    mapping = {}
    with csv_path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # skip header lines
            if line.lower().startswith("file"):
                continue
            # allow ';' delimiter between file and actions
            if ";" in line:
                file_part, action_part = line.split(";", 1)
            else:
                continue
            filename = file_part.strip()
            actions = normalize_actions(action_part)
            mapping[filename] = actions
    return mapping

def category_for_clip(is_violent_folder: bool, actions: set[str]) -> str:
    """
    Decide which of the 3 folders:
      - high-level violence
      - low-level violence
      - non-violence
    """
    if not is_violent_folder:
        # non-violent folder clips go to non-violence
        return "non-violence"

    # violent folder: use actions to split high vs low
    if actions & HIGH_LEVEL_LABELS:
        return "high-level violence"
    if actions & LOW_LEVEL_LABELS:
        return "low-level violence"

    # fallback: if violent folder but no matched label, put into low-level (safer)
    return "low-level violence"

def ensure_out_dirs():
    for cat in ["high-level violence", "low-level violence", "non-violence"]:
        for cam in ["cam1", "cam2"]:
            (OUT / cat / cam).mkdir(parents=True, exist_ok=True)

def copy_or_move(src: Path, dst: Path):
    if COPY_FILES:
        shutil.copy2(src, dst)
    else:
        shutil.move(src, dst)

def main():
    ensure_out_dirs()

    nonviolent_map = read_csv_map(CSV_NONVIOLENT)
    violent_map    = read_csv_map(CSV_VIOLENT)

    # process both top folders
    for top in ["non-violent", "violent"]:
        is_violent = (top == "violent")
        for cam in ["cam1", "cam2"]:
            src_dir = ROOT / top / cam
            if not src_dir.exists():
                print(f"[WARN] Missing folder: {src_dir}")
                continue

            for mp4 in sorted(src_dir.glob("*.mp4")):
                actions = (violent_map if is_violent else nonviolent_map).get(mp4.name, set())
                cat = category_for_clip(is_violent, actions)

                dst = OUT / cat / cam / mp4.name
                copy_or_move(mp4, dst)

    print("Done!")
    print("Output at:", OUT)

    print("CWD:", Path.cwd())
    print("BASE_DIR:", BASE_DIR)
    print("ROOT:", ROOT)
    print("OUT:", OUT)

if __name__ == "__main__":
    main()


Done!
Output at: /content/data/processed/violence-detection-dataset
CWD: /content
BASE_DIR: /content
ROOT: /content/data/raw/violence-detection-dataset
OUT: /content/data/processed/violence-detection-dataset


### 2. Divide each video into chunks with 16 frames at a resolution of 112 x 112 (e.g. a video may have 8 chunks) and store the samples and labels to data\processed\violence-detection-dataset\airtlabDataset

In [11]:
# Utility functions for the experiments (chunk count, video preprocessing, feature computation, )

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import csv


def count_chunks(videoBasePath):
    """Counts the 16 frames lenght chunks available in a dataset organized in violent and non-violent,
    cam1 and cam2 folders, placed at videoBasePath.

    Parameters
    ----------
    videoBasePath : str
                    Base path of the dataset

    Returns
    -------
    cnt : int
          number of 16 frames lenght chunks in the dataset
    """

    folders = ['non-violence', 'low-level violence', 'high-level violence']
    cams = ['cam1', 'cam2']
    cnt = 0

    for folder in folders:
        for camName in cams:
            path = os.path.join(videoBasePath, folder, camName)

            videofiles = os.listdir(path)
            for videofile in videofiles:
                filePath = os.path.join(path, videofile)
                video = cv2.VideoCapture(filePath)
                numframes = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
                fps = int(video.get(cv2.CAP_PROP_FPS))
                chunks = numframes//16
                cnt += chunks


    return cnt

def preprocessVideos(videoBasePath, featureBasePath, verbose=True):
    """Preproccess all the videos.

    It extracts samples for the input of C3D from a video dataset, organised in violent and non-violent, cam1 and cam2 folders.
    The samples and the labels are store on two memmap numpy arrays, called samples.mmap and labels.mmap, at "featureBasePath".
    The numpy array with samples has shape (Chunk #, 16, 112, 112, 3), the labels array has shape (Chunk # 16, 112, 112, 3).
    For the AIRTLab dataset the number of chunks is 3537.

    Parameters
    ----------
    videoBasePath : str
                    Pathname to the base of the video repository, which contains two directories,
                    violent and non-violent, which are divided into cam1 and cam2.
    featureBasePath : str
                      it is the pathname of a base where the numpy arrays have to be saved.
    verbose : bool
              if True print debug logs (default True)

    """
    folders = ['non-violence', 'low-level violence', 'high-level violence']
    cams = ['cam1', 'cam2']
    total_chunks = count_chunks(videoBasePath)
    npSamples = np.memmap(os.path.join(featureBasePath, 'samples.mmap'), dtype=np.float32, mode='w+', shape=(total_chunks, 16, 112, 112, 3))
    npLabels = np.memmap(os.path.join(featureBasePath, 'labels.mmap'), dtype=np.int8, mode='w+', shape=(total_chunks))
    npVideos = np.memmap(os.path.join(featureBasePath, 'videos.mmap'), dtype=np.int32, mode='w+', shape=(total_chunks))
    cnt = 0

    for folder in folders:
        for camName in cams:
            path = os.path.join(videoBasePath, folder, camName)

            videofiles = os.listdir(path)
            for videofile in videofiles:
                filePath = os.path.join(path, videofile)
                video = cv2.VideoCapture(filePath)
                numframes = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
                fps = int(video.get(cv2.CAP_PROP_FPS))
                chunks = numframes//16
                if verbose:
                    print("*** [Video Info] Number of frames: {} - fps: {} - chunks: {}".format(numframes, fps, chunks))
                vid = []
                videoFrames = []
                while True:
                    ret, img = video.read()
                    if not ret:
                        break
                    videoFrames.append(cv2.resize(img, (112, 112)))
                vid = np.array(videoFrames, dtype=np.float32)
                filename = os.path.splitext(videofile)[0]
                chunk_cnt = 0
                for i in range(chunks):
                    X = vid[i*16:i*16+16]
                    chunk_cnt += 1
                    npSamples[cnt] = np.array(X, dtype=np.float32)


                    if folder == 'high-level violence':
                        npLabels[cnt] = np.int8(2)
                    elif folder == 'low-level violence':
                        npLabels[cnt] = np.int8(1)
                    else:
                        npLabels[cnt] = np.int8(0)
                    npVideos[cnt] = np.int8(filename)

                    cnt += 1

    if verbose:
        print("** Labels **")
        print(npLabels.shape)
        print('\n****\n')
        print("** Samples **")
        print(npSamples.shape)
        print('\n****\n')

    del npSamples
    del npLabels
    del npVideos


In [14]:
OUT = BASE_DIR / "data" / "processed" / "violence-detection-dataset"
preprocessVideos(OUT, OUT, True)

*** [Video Info] Number of frames: 158 - fps: 30 - chunks: 9
*** [Video Info] Number of frames: 159 - fps: 30 - chunks: 9
*** [Video Info] Number of frames: 159 - fps: 30 - chunks: 9
*** [Video Info] Number of frames: 128 - fps: 30 - chunks: 8
*** [Video Info] Number of frames: 261 - fps: 30 - chunks: 16
*** [Video Info] Number of frames: 150 - fps: 30 - chunks: 9
*** [Video Info] Number of frames: 369 - fps: 30 - chunks: 23
*** [Video Info] Number of frames: 159 - fps: 30 - chunks: 9
*** [Video Info] Number of frames: 231 - fps: 30 - chunks: 14
*** [Video Info] Number of frames: 192 - fps: 30 - chunks: 12
*** [Video Info] Number of frames: 74 - fps: 30 - chunks: 4
*** [Video Info] Number of frames: 138 - fps: 30 - chunks: 8
*** [Video Info] Number of frames: 101 - fps: 30 - chunks: 6
*** [Video Info] Number of frames: 153 - fps: 30 - chunks: 9
*** [Video Info] Number of frames: 136 - fps: 30 - chunks: 8
*** [Video Info] Number of frames: 177 - fps: 30 - chunks: 11
*** [Video Info] Num